# RAG Chat Main — Naive Combined (Dukcapil + OPD)

Naive RAG pipeline yang query **DUA vector store sekaligus** dalam satu pass, **tanpa router**.

**Posisi di project:**
- `rag_chat.ipynb` → dukcapil only (4 variants, sudah ada)
- `rag_chat_opd.ipynb` → OPD only (3 variants, sudah ada)
- **`rag_chat_main.ipynb` ← INI: combined, no routing**
- `agenticrag/2-agentic_router.ipynb` → agentic dgn router (sudah ada)

**Strategi:**
- Retrieve hybrid (BM25 + dense, RRF, weights 0.5/0.5) dari **masing-masing store**
- `k_per_store=4` → total 8 docs (match agentic-both: 4+4 = 8)
- Tag tiap doc dgn `metadata._source` (`dukcapil` atau `opd`)
- Generate pakai `PROMPT_COMBINED` yang generic (tidak force struktur 2-bagian)

**Hipotesis latency vs agentic:**
- ✅ Hilang router overhead (~2.4s LLM call)
- ❌ Selalu retrieve 2 store (boros buat query single-domain atau off-topic)
- ❌ Selalu generate dgn context 2x (~8 docs)
- **Net**: di dataset kecil (150+61 docs), kemungkinan naive combined MENANG karena router overhead lebih besar dari ekstra retrieve cost.

## Step 1 — Import module

In [ ]:
from ragtrial.capabilities import CAPABILITIES, format_context as _format_context
from ragtrial.llm import embeddings, llm
from ragtrial.rag.naive_combined import ask_main, retrieve_combined
from ragtrial.rag.prompts import PROMPT_COMBINED

# Force eager init so legacy notebook cells can grab the raw Chroma / docs handles.
CAPABILITIES["dukcapil"]._ensure_initialized()
CAPABILITIES["opd"]._ensure_initialized()
vs_dukcapil = CAPABILITIES["dukcapil"]._vectorstore
vs_opd = CAPABILITIES["opd"]._vectorstore
dukcapil_docs_all = CAPABILITIES["dukcapil"]._all_docs
opd_docs_all = CAPABILITIES["opd"]._all_docs

def retrieve_dukcapil_hybrid(q, k=4):
    return CAPABILITIES["dukcapil"].invoke(q, k=k)

def retrieve_opd_hybrid(q, k=4):
    return CAPABILITIES["opd"].invoke(q, k=k)

def format_context_combined(docs):
    return _format_context(docs, CAPABILITIES)

print(f"✓ Module loaded")
print(f"   Dukcapil docs in-memory: {len(dukcapil_docs_all)}")
print(f"   OPD docs in-memory:      {len(opd_docs_all)}")

## Step 2 — Inspect retrieval (sanity check)

Cek dulu sebelum generate: untuk query single-domain, apakah retrieval dari store yang TIDAK relevan menghasilkan docs yang ke-mix (noisy)? Ini penting untuk evaluasi kualitatif nanti.

In [ ]:
inspect_queries = [
    ("Apa syarat KTP-el?", "dukcapil-only"),
    ("Alamat Dinas Pariwisata Batang?", "opd-only"),
    ("Cara bikin akta kelahiran dan ke dinas mana?", "both"),
]
for q, expected in inspect_queries:
    docs = retrieve_combined(q, k_per_store=4)
    print(f"\nQ: {q}  (expected: {expected})")
    for i, d in enumerate(docs, 1):
        src = d.metadata.get("_source")
        if src == "opd":
            label = f"OPD-{d.metadata.get('nama_opd', '?')[:40]}"
        else:
            label = f"DUK-{d.metadata.get('section', '?')[:30]} hal{d.metadata.get('page', '?')}"
        snippet = d.page_content[:80].replace("\n", " ")
        print(f"   [{i:2d}] {src:8s} | {label:50s} | {snippet}...")

## Step 3 — Smoke test 6 query (end-to-end + latency)

Mix coverage: dukcapil-only, opd-only, both, dan satu out-of-scope.

In [ ]:
TEST_QUERIES = [
    # dukcapil-only
    "Apa syarat penerbitan KTP-el pertama kali bagi WNI?",
    "Bagaimana prosedur pindah domisili untuk WNA pemegang KITAP?",
    # opd-only
    "Alamat Dinas Pariwisata Kabupaten Batang di mana?",
    "Nomor telepon Sekretariat Daerah Batang berapa?",
    # both
    "Mau urus akta kematian, ke dinas mana dan apa syaratnya?",
    # off-topic
    "Resep nasi goreng spesial?",
]

results = []
for q in TEST_QUERIES:
    r = ask_main(q, verbose=True)
    results.append(r)

## Step 4 — Latency summary

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {
        "query": r["question"][:55] + ("..." if len(r["question"]) > 55 else ""),
        "retrieve_s": round(r["timings"]["retrieve"], 2),
        "generate_s": round(r["timings"]["generate"], 2),
        "total_s":    round(r["timings"]["total"], 2),
    }
    for r in results
])
print(summary.to_string(index=False))
print(f"\nMean total: {summary['total_s'].mean():.2f}s")
print(f"Mean retrieve: {summary['retrieve_s'].mean():.2f}s")
print(f"Mean generate: {summary['generate_s'].mean():.2f}s")

## Step 5 — Observasi

Yang perlu dicatat manual setelah run:
1. **Apakah retrieval ke-mix?** Untuk query dukcapil-only (mis. KTP-el), apakah ada docs OPD yang ikut top-4? Kalau ya, apakah mengganggu jawaban?
2. **Generate behavior off-topic**: untuk "resep nasi goreng", apakah model konsisten jawab "informasi tidak ditemukan" sesuai aturan 7 di `PROMPT_COMBINED`?
3. **Latency vs agentic**: bandingin mean total di sini vs ~7.2s agentic (dari notebook 3 sebelumnya). Hipotesis menang/kalah?

Comparison head-to-head lengkap → lihat `agenticrag/3-compare_agentic_vs_naive.ipynb`.